# SAM Food Segmentation - Inference Demo

This notebook demonstrates how to use the SAM model with LoRA adaptation for food segmentation inference.
It uses the existing `SAMLoRAModel` class but patches the internal SAM instance to allow using the standard `SamPredictor` for inference.

In [1]:
import os
import sys
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.append(os.getcwd())

from configs.config import Config
from src.models.sam_lora import SAMLoRAModel
from segment_anything import SamPredictor

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

## 1. Configuration and Model Setup

First, we need to set up the configuration and load the model.
**Note**: You need to provide the path to the base SAM checkpoint (e.g., `sam_vit_b_01ec64.pth`).

In [ ]:
# Initialize config
config = Config()

# Set model paths
# TODO: Set the path to your base SAM checkpoint (required)
# Download from: https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
config.model.sam_checkpoint_path = "sam_vit_b_01ec64.pth" 

# TODO: Set the path to your trained LoRA checkpoint (optional)
lora_checkpoint_path = "checkpoints/best_model.pth"

config.system.device = device

In [ ]:
def load_model(config, lora_checkpoint_path=None):
    # Initialize model structure
    model = SAMLoRAModel(config)
    model.to(device)
    
    # Load LoRA weights if provided
    if lora_checkpoint_path and os.path.exists(lora_checkpoint_path):
        print(f"Loading LoRA checkpoint from {lora_checkpoint_path}...")
        checkpoint = torch.load(lora_checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print("LoRA checkpoint loaded successfully.")
    else:
        print("No LoRA checkpoint found or provided. Using base SAM model.")
    
    # PATCH: Replace the original SAM components with the LoRA-adapted ones
    # This allows us to use the standard SamPredictor with our LoRA model
    model._original_sam_model.mask_decoder = model.mask_decoder
    model._original_sam_model.prompt_encoder = model.prompt_encoder
    
    return model

# Create model instance
try:
    model_wrapper = load_model(config, lora_checkpoint_path)
    
    # Use the standard SamPredictor with the patched internal model
    predictor = SamPredictor(model_wrapper._original_sam_model)
    print("Model and Predictor initialized successfully!")
except Exception as e:
    print(f"Error initializing model: {e}")
    print("Please ensure you have downloaded the base SAM checkpoint.")

## 2. Load and Prepare Image

In [ ]:
def show_image(image, points=None, labels=None, box=None, mask=None, ax=None, title=None):
    if ax is None:
        plt.figure(figsize=(10, 10))
        ax = plt.gca()
    
    ax.imshow(image)
    
    if mask is not None:
        show_mask(mask, ax)
        
    if points is not None:
        pos_points = points[labels==1]
        neg_points = points[labels==0]
        ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=200, edgecolor='white', linewidth=1.25, label='Positive')
        ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=200, edgecolor='white', linewidth=1.25, label='Negative')
        
    if box is not None:
        x0, y0, x1, y1 = box
        w, h = x1 - x0, y1 - y0
        ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2, label='Box'))
        
    if title:
        ax.set_title(title)
    ax.axis('on')

def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

In [ ]:
# Load test image
image_path = "test_image/test_image.jpg"

if os.path.exists(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.title("Test Image")
    plt.axis('on')
    plt.show()
else:
    print(f"Image not found at {image_path}")
    # Create a dummy image if file doesn't exist
    image = np.zeros((512, 512, 3), dtype=np.uint8)
    cv2.circle(image, (256, 256), 100, (255, 100, 100), -1)

## 3. Run Inference

Define prompts (points or boxes) and run prediction.

In [ ]:
# Set image embedding
predictor.set_image(image)

# Define prompts
# Example: Point at the center of the image
h, w = image.shape[:2]
input_point = np.array([[w//2, h//2]])
input_label = np.array([1]) # 1 indicates a foreground point

# Example: Bounding box (optional)
# input_box = np.array([w//4, h//4, w*3//4, h*3//4])

print(f"Prompts: Point {input_point}, Label {input_label}")

In [ ]:
# Predict
masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True # SAM can output multiple masks for a single prompt
)

print(f"Generated {len(masks)} masks with scores: {scores}")

## 4. Visualize Results

In [ ]:
# Visualize masks
for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.figure(figsize=(10, 10))
    show_image(image, input_point, input_label, title=f"Mask {i+1}, Score: {score:.3f}")
    show_mask(mask, plt.gca())
    plt.show()